# FLARE LAI experiment grid (`flare_lai_exp`)

Pull the Terra data table
[`flare_lai_exp`](https://app.terra.bio/#workspaces/allofus-drc-wgs-LR-prodData/AoU_DRC_LongReads_PhaseTwo_Storage)
via Firecloud/FISS, then score every **finished** row with
`scripts/flare_switch_qc.py` on that row's `region`.

Switch / tract metrics below are **diagnostics**. Recipe selection uses the
Part 8 two-stage association-facing rule (concordance + Mendelian gates →
Tractor null-λ winner). See `flare/flare_lai_fix_instructions.md` Part 8.

Diagnostic checklist (not the decision rule):

1. **Flicker rate** (`flicker / hap / Mb`) low alongside **flicker fraction** (target ≪20%, ideally ~5% for AFR/AMR)
2. **Switches / hap / Mb** near pinned T with `prop_*`-adjusted expectation (≈0.08 for T=8, ≈0.12 for T=12 when heterozygosity factor ≈1)
3. **µ** still AA/Latino-like when models are present
4. **Span** long enough for pinned T (`span_ok`); warn when the window is too short

Re-run this notebook as more rows finish; incomplete rows are skipped.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook

SCRIPTS = init_notebook(
    "workspace_paths.py", "flare_switch_qc.py", "flare_lai_exp.py", "flare_model.py", "flare_build_af_panel.py", "flare_score_allele_ancestry.py", "flare_score_mendelian_lai.py", "flare_lai_null_lambda.py"
)
from workspace_paths import data_root

ROOT = data_root()
OUT = ROOT / "flare_lai_exp"
OUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUT:", OUT)
print("scripts:", SCRIPTS)

## Config

Defaults target the storage workspace. Override with env vars or edit this cell.
Set `FORCE_RESCAN=true` to re-run switch QC after a CLI change.

In [ ]:
import os
from pathlib import Path

TERRA_NAMESPACE = os.environ.get("TERRA_NAMESPACE", "allofus-drc-wgs-LR-prodData")
TERRA_WORKSPACE = os.environ.get(
    "TERRA_WORKSPACE", "AoU_DRC_LongReads_PhaseTwo_Storage"
)
ENTITY_TYPE = os.environ.get("FLARE_LAI_EXP_ENTITY", "flare_lai_exp")
TABLE_TSV = os.environ.get("FLARE_LAI_EXP_TSV", "").strip()

COVARIATES = os.environ.get(
    "COVARIATES",
    "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/covariates/covariates.source_rebuilt.csv.gz",
).strip()

FLICKER_MAX_BP = int(os.environ.get("FLICKER_MAX_BP", "50000"))
BG_KEEP_EVERY = int(os.environ.get("BG_KEEP_EVERY", "50"))
FORCE_RESCAN = os.environ.get("FORCE_RESCAN", "false").strip().lower() in {
    "1", "true", "yes", "on",
}
# Fast path for grid scoring: AN1/AN2 rates only, skip fat TSVs, shard samples.
# Default jobs ≈ nproc-2 (this VM has 32 CPUs). Override with SWITCH_QC_JOBS.
_nproc = os.cpu_count() or 4
SWITCH_QC_JOBS = int(os.environ.get("SWITCH_QC_JOBS", str(max(1, _nproc - 2))))
SWITCH_QC_AN_ONLY = os.environ.get("SWITCH_QC_AN_ONLY", "true").strip().lower() in {
    "1", "true", "yes", "on",
}
SWITCH_QC_SUMMARY_ONLY = os.environ.get(
    "SWITCH_QC_SUMMARY_ONLY", "true"
).strip().lower() in {"1", "true", "yes", "on"}
# Optional: comma list of entity ids to score; empty = all complete rows
ONLY_IDS = [
    x.strip()
    for x in os.environ.get("FLARE_LAI_EXP_ONLY", "").split(",")
    if x.strip()
]

print("workspace:", f"{TERRA_NAMESPACE}/{TERRA_WORKSPACE}")
print("entity:", ENTITY_TYPE)
print("TABLE_TSV:", TABLE_TSV or "(firecloud)")
print("FORCE_RESCAN:", FORCE_RESCAN)
print("SWITCH_QC_JOBS:", SWITCH_QC_JOBS, f"(nproc={_nproc})")
print("SWITCH_QC_AN_ONLY:", SWITCH_QC_AN_ONLY)
print("SWITCH_QC_SUMMARY_ONLY:", SWITCH_QC_SUMMARY_ONLY)
print("ONLY_IDS:", ONLY_IDS or "(all complete)")
print("OUT:", OUT)

## Fetch `flare_lai_exp`

Uses `firecloud.api.get_entities_tsv` / `get_entities` (FISS) unless
`FLARE_LAI_EXP_TSV` points at an export.

In [ ]:
from flare_lai_exp import (
    DEFAULT_ID_COLUMN,
    fetch_lai_exp_table,
    row_is_complete,
    status_frame,
)

if TABLE_TSV:
    print("table TSV:", TABLE_TSV)
else:
    print(f"firecloud get_entities {TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")

exp = fetch_lai_exp_table(
    tsv=TABLE_TSV or None,
    from_firecloud=not TABLE_TSV,
    namespace=TERRA_NAMESPACE,
    workspace=TERRA_WORKSPACE,
    entity_type=ENTITY_TYPE,
)
id_col = DEFAULT_ID_COLUMN if DEFAULT_ID_COLUMN in exp.columns else [
    c for c in exp.columns if str(c).startswith("entity:") or str(c).endswith("_id")
][0]
if id_col != DEFAULT_ID_COLUMN:
    exp = exp.rename(columns={id_col: DEFAULT_ID_COLUMN})
    id_col = DEFAULT_ID_COLUMN

status = status_frame(exp, id_column=id_col)
table_path = OUT / "flare_lai_exp.table.tsv"
status_path = OUT / "flare_lai_exp.status.tsv"
exp.to_csv(table_path, sep="\t", index=False)
status.to_csv(status_path, sep="\t", index=False)
print(f"rows={len(exp)} complete={int(status['complete'].sum())}")
print("wrote", table_path)
print("wrote", status_path)
display(status)

## Localize + switch-QC finished rows

For each complete row: copy `anc_vcf` (+ `.tbi`) and `models_tsv`, run
`flare_switch_qc.py --region <row.region>` with `--jobs` / `--an-only` /
`--summary-only` (see config). Skips rows whose `summary.json` already exists
unless `FORCE_RESCAN`.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd

from flare_lai_exp import load_models_tsv, row_is_complete, summarize_models, tract_metrics_from_summary
from flare_switch_qc import implied_T_given_props, span_ok_for_t


def _localize(uri: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0 and not FORCE_RESCAN:
        return dest
    print("gsutil cp", uri, dest)
    subprocess.check_call(["gsutil", "cp", uri, str(dest)])
    if uri.endswith(".vcf.gz") or uri.endswith(".bcf"):
        tbi = dest.with_name(dest.name + (".tbi" if uri.endswith(".vcf.gz") else ".csi"))
        # Prefer sibling index from table when present; else try URI + .tbi
        idx_uri = uri + (".tbi" if uri.endswith(".vcf.gz") else ".csi")
        try:
            subprocess.check_call(["gsutil", "cp", idx_uri, str(tbi)])
        except subprocess.CalledProcessError:
            print("warning: no index at", idx_uri)
    return dest


def _run_switch_qc(vcf: Path, out_dir: Path, region: str) -> dict:
    summary_path = out_dir / "summary.json"
    if summary_path.is_file() and not FORCE_RESCAN:
        return json.loads(summary_path.read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(SCRIPTS / "flare_switch_qc.py"),
        "--vcf", str(vcf),
        "--out-dir", str(out_dir),
        "--flicker-max-bp", str(FLICKER_MAX_BP),
        "--bg-keep-every", str(BG_KEEP_EVERY),
        "--jobs", str(SWITCH_QC_JOBS),
    ]
    if SWITCH_QC_AN_ONLY:
        cmd.append("--an-only")
    if SWITCH_QC_SUMMARY_ONLY:
        cmd.append("--summary-only")
    if region:
        cmd.extend(["--region", region])
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    return json.loads(summary_path.read_text())


compare_rows = []
model_rows = []

for _, row in exp.iterrows():
    eid = str(row[id_col])
    if ONLY_IDS and eid not in ONLY_IDS:
        continue
    if not row_is_complete(row):
        print("skip (incomplete):", eid)
        continue

    run_dir = OUT / eid
    run_dir.mkdir(parents=True, exist_ok=True)
    local_vcf = _localize(str(row["anc_vcf"]), run_dir / Path(str(row["anc_vcf"])).name)
    # Prefer table index when present
    if str(row.get("anc_vcf_index") or "").startswith("gs://"):
        _localize(str(row["anc_vcf_index"]), Path(str(local_vcf) + ".tbi"))
    local_models = _localize(
        str(row["models_tsv"]), run_dir / Path(str(row["models_tsv"])).name
    )

    region = str(row.get("region") or "").strip()
    qc_dir = run_dir / "switch_qc"
    summary = _run_switch_qc(local_vcf, qc_dir, region)
    metrics = tract_metrics_from_summary(summary)

    models = load_models_tsv(local_models)
    display(models.assign(experiment=eid))
    summ = summarize_models(models, row)
    for m in summ:
        model_rows.append({"experiment": eid, **m})

    pinned_ts = [m["pinned_gen"] for m in summ if m.get("pinned_gen") is not None]
    span_mb = metrics.get("span_mb")
    span_ok = None
    if pinned_ts and span_mb is not None:
        span_ok = all(span_ok_for_t(span_mb, t) for t in pinned_ts)
        if span_ok is False:
            print(
                f"warning: {eid} span_mb={span_mb} too short for pinned T "
                f"(need >=3 mean tracts; min T={min(pinned_ts)})"
            )

    props_lists = [m["props"] for m in summ if m.get("props")]
    pooled_props = None
    if props_lists:
        n_anc = len(props_lists[0])
        pooled_props = [
            sum(p[i] for p in props_lists) / len(props_lists) for i in range(n_anc)
        ]

    obs_rate = metrics.get("switches_per_hap_per_mb")
    implied_t = (
        implied_T_given_props(obs_rate, pooled_props)
        if pooled_props and obs_rate is not None
        else metrics.get("implied_t_gen")
    )

    compare_rows.append(
        {
            "experiment": eid,
            "notes": row.get("notes", ""),
            "em": row.get("em", ""),
            "gen": row.get("gen", ""),
            "gen_by_pop": row.get("gen_by_pop", ""),
            "include_pops": row.get("include_pops", ""),
            "min_mac": row.get("min_mac", ""),
            "min_maf": row.get("min_maf", ""),
            "region": region,
            **metrics,
            "span_ok": span_ok,
            "implied_t_gen": implied_t,
        }
    )

cmp = pd.DataFrame(compare_rows)
models_long = pd.DataFrame(model_rows)
if not cmp.empty:
    cmp_path = OUT / "lai_exp_switch_summary.tsv"
    cmp.to_csv(cmp_path, sep="\t", index=False)
    print("wrote", cmp_path)
    display(cmp)
else:
    print("No complete rows to score yet.")

if not models_long.empty:
    models_path = OUT / "lai_exp_models_long.tsv"
    models_long.to_csv(models_path, sep="\t", index=False)
    print("wrote", models_path)
    display(models_long)

## Plots

Per-pop pinned expectations (with `prop_*` heterozygosity factor) come from
`lai_exp_by_pop.tsv`. Black ticks on the rate plot show the mean per-pop
expected rate when by-pop rows exist. `rate_over_expected` is monotone in T —
it falsifies a pinned T, it does not select one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if cmp.empty:
    print("skip plots: no scored experiments")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    x = np.arange(len(cmp))
    labels = cmp["experiment"].tolist()

    axes[0, 0].bar(x, cmp["switches_per_hap_per_mb"], color="#4C78A8")
    if not models_long.empty:
        exp_by_eid = models_long.groupby("experiment")["expected_rate_pinned"].mean()
        for i, eid in enumerate(labels):
            if eid in exp_by_eid.index and pd.notna(exp_by_eid[eid]):
                axes[0, 0].plot(
                    [i, i],
                    [exp_by_eid[eid], exp_by_eid[eid]],
                    color="black",
                    marker="_",
                    markersize=14,
                )
    axes[0, 0].set_xticks(x, labels, rotation=30, ha="right")
    axes[0, 0].set_ylabel("switches / hap / Mb")
    axes[0, 0].set_title("Observed rate vs per-pop pinned expectation")

    axes[0, 1].bar(x, cmp["flicker_frac"], color="#F58518")
    axes[0, 1].axhline(0.2, color="gray", ls="--", lw=1)
    axes[0, 1].set_xticks(x, labels, rotation=30, ha="right")
    axes[0, 1].set_ylabel("flicker fraction")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].set_title("A→B→A within 50 kb (of switches)")

    axes[1, 0].bar(x, cmp["flicker_per_hap_per_mb"], color="#E45756")
    axes[1, 0].set_xticks(x, labels, rotation=30, ha="right")
    axes[1, 0].set_ylabel("flicker / hap / Mb")
    axes[1, 0].set_title("Flicker rate (checklist metric)")

    if not models_long.empty:
        ax = axes[1, 1]
        for pop, sub in models_long.groupby("population"):
            ax.scatter(sub["experiment"], sub["model_t_gen"], label=pop, s=60)
        ax.tick_params(axis="x", rotation=30)
        ax.set_ylabel("model t_gen")
        ax.set_title("Per-pop EM / output T")
        ax.legend(frameon=False, fontsize=8)
    else:
        axes[1, 1].axis("off")

    fig.tight_layout()
    fig_path = OUT / "lai_exp_compare.png"
    fig.savefig(fig_path, dpi=150)
    print("wrote", fig_path)
    plt.show()

## Per-population rates (finished multi-pop rows)

Uses covariates `population` when the local/GCS covariates file is available.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd

from flare_lai_exp import pinned_t_for_pop, row_is_complete
from flare_switch_qc import expected_switches_per_hap_per_mb, implied_T_given_props, span_ok_for_t

cov_local = OUT / "covariates.source_rebuilt.csv.gz"
pop_map = None
if COVARIATES:
    if COVARIATES.startswith("gs://"):
        if not cov_local.is_file():
            print("gsutil cp", COVARIATES, cov_local)
            subprocess.check_call(["gsutil", "cp", COVARIATES, str(cov_local)])
        cov_path = cov_local
    else:
        cov_path = Path(COVARIATES)
    if cov_path.is_file():
        pop_map = pd.read_csv(cov_path, usecols=["research_id", "population"], dtype=str)
        print("covariates pops:", sorted(pop_map["population"].dropna().unique()))
    else:
        print("covariates missing:", cov_path)

models_by_key = (
    models_long.set_index(["experiment", "population"]) if not models_long.empty else None
)

by_pop_rows = []
if pop_map is None:
    print("skip by-pop: no covariates")
elif cmp.empty:
    print("skip by-pop: no scored experiments")
else:
    for _, row in exp.iterrows():
        eid = str(row[id_col])
        if ONLY_IDS and eid not in ONLY_IDS:
            continue
        if not row_is_complete(row):
            continue
        qc_dir = OUT / eid / "switch_qc"
        slim_path = qc_dir / "switches_slim.tsv.gz"
        sw_path = qc_dir / "switches.tsv.gz"
        summary_path = qc_dir / "summary.json"
        if not summary_path.is_file():
            continue
        if slim_path.is_file():
            sw_use = slim_path
        elif sw_path.is_file():
            sw_use = sw_path
        else:
            continue
        summary = json.loads(summary_path.read_text())
        span = (summary.get("tracts") or {}).get("span_mb") or 1.0
        sw = pd.read_csv(sw_use, sep="\t", dtype={"sample": str})
        vcf = next((OUT / eid).glob("*.anc.vcf.gz"), None)
        if vcf is None:
            print("skip by-pop (no local anc vcf):", eid)
            continue
        vcf_samples = subprocess.check_output(
            ["bcftools", "query", "-l", str(vcf)], text=True
        ).splitlines()
        samples = pd.DataFrame({"sample": vcf_samples})
        samples = samples.merge(pop_map, left_on="sample", right_on="research_id", how="left")
        sw = sw.merge(pop_map, left_on="sample", right_on="research_id", how="left")
        sw["population"] = sw["population"].fillna("NA")
        n_samp = samples.groupby(samples["population"].fillna("NA"))["sample"].nunique()
        n_sw = sw.groupby("population").size()
        n_fl = sw.groupby("population")["is_flicker"].sum() if "is_flicker" in sw.columns else 0
        for pop in sorted(set(n_samp.index) | set(n_sw.index)):
            ns = int(n_samp.get(pop, 0))
            nsw = int(n_sw.get(pop, 0))
            nfl = int(n_fl.get(pop, 0)) if hasattr(n_fl, "get") else 0
            rate = nsw / (ns * 2) / span if ns else None
            flicker_rate = nfl / (ns * 2) / span if ns else None
            pinned = pinned_t_for_pop(row, pop)
            props = None
            expected = None
            if models_by_key is not None and (eid, pop) in models_by_key.index:
                mrow = models_by_key.loc[(eid, pop)]
                props = mrow.get("props")
                expected = mrow.get("expected_rate_pinned")
            if expected is None and pinned is not None:
                expected = expected_switches_per_hap_per_mb(pinned, props)
            implied_t = (
                implied_T_given_props(rate, props)
                if props and rate is not None
                else None
            )
            span_ok = span_ok_for_t(span, pinned)
            by_pop_rows.append(
                {
                    "experiment": eid,
                    "population": pop,
                    "n_samples": ns,
                    "n_switches": nsw,
                    "n_flicker": nfl,
                    "flicker_frac": (nfl / nsw) if nsw else None,
                    "flicker_per_hap_per_mb": flicker_rate,
                    "switches_per_hap_per_mb": rate,
                    "pinned_gen": pinned,
                    "expected_rate": expected,
                    "implied_t_gen": implied_t,
                    "span_ok": span_ok,
                }
            )
            if span_ok is False:
                print(
                    f"warning: {eid}/{pop} span_mb={span} too short for pinned T={pinned}"
                )

by_pop = pd.DataFrame(by_pop_rows)
if not by_pop.empty:
    # rate_over_expected is monotone in T — falsifies, does not select T
    by_pop["rate_over_expected"] = by_pop["switches_per_hap_per_mb"] / by_pop["expected_rate"]
    by_pop_path = OUT / "lai_exp_by_pop.tsv"
    by_pop.to_csv(by_pop_path, sep="\t", index=False)
    print("wrote", by_pop_path)
    display(by_pop)
else:
    print("no by-pop rows")

## Diagnostics: global ancestry proportions from AN1/AN2

Integrate per-site AN1/AN2 calls across haps and compare to model `prop_*`
(from `models.tsv`, not the mislabeled legacy `mu_*` columns) and to cohort
ancestry priors in covariates when available.

In [ ]:
import json
from collections import Counter

from flare_switch_qc import ANCESTRY

# TODO: wire soft admixture probability columns from rebuilt covariates when
# named in covariates.source_rebuilt.data_dictionary.tsv (no afr_prob/eur_prob
# columns today — only hard population / ancestry_pred* labels).

anc_diag_rows = []
cov_prop_cols = []
if COVARIATES:
    cov_path = (
        OUT / "covariates.source_rebuilt.csv.gz"
        if COVARIATES.startswith("gs://")
        else Path(COVARIATES)
    )
    if cov_path.is_file():
        cov_hdr = pd.read_csv(cov_path, nrows=0).columns.tolist()
        cov_prop_cols = [
            c
            for c in cov_hdr
            if c.startswith("prop_") or c.endswith("_prob") or "admix" in c.lower()
        ]
        if not cov_prop_cols:
            print(
                "TODO: set covariate admixture-probability column names in this cell "
                "(rebuilt covariates have population / ancestry_pred*, not prop_*)."
            )


def global_props_from_vcf(vcf_path: Path, region: str = "") -> dict[str, float]:
    cmd = ["bcftools", "query"]
    if region:
        cmd.extend(["-r", region])
    cmd.extend(["-f", "[%AN1][%AN2]\n", str(vcf_path)])
    text = subprocess.check_output(cmd, text=True)
    counts: Counter[int] = Counter()
    for line in text.splitlines():
        for field in line.split("\t"):
            for anc_s in field.split(","):
                anc_s = anc_s.strip()
                if not anc_s or anc_s == ".":
                    continue
                try:
                    counts[int(float(anc_s))] += 1
                except ValueError:
                    pass
    total = sum(counts.values())
    if not total:
        return {}
    return {ANCESTRY.get(k, str(k)): counts[k] / total for k in sorted(counts)}


if cmp.empty:
    print("skip AN props: no scored experiments")
else:
    for _, row in exp.iterrows():
        eid = str(row[id_col])
        if ONLY_IDS and eid not in ONLY_IDS:
            continue
        if not row_is_complete(row):
            continue
        run_dir = OUT / eid
        vcf = next(run_dir.glob("*.anc.vcf.gz"), None)
        region = str(row.get("region") or "").strip()
        if vcf is None:
            print("skip AN props (no local anc vcf):", eid)
            continue
        global_props = global_props_from_vcf(vcf, region)
        models_path = run_dir / Path(str(row["models_tsv"])).name
        model_props = {}
        if models_path.is_file():
            mdf = load_models_tsv(models_path)
            prop_cols = [c for c in mdf.columns if str(c).startswith("prop_")]
            if prop_cols and not mdf.empty:
                means = mdf[prop_cols].astype(float).mean()
                model_props = {
                    c.replace("prop_", ""): float(means[c]) for c in prop_cols
                }
        for anc, frac in sorted(global_props.items()):
            anc_diag_rows.append(
                {
                    "experiment": eid,
                    "ancestry": anc,
                    "global_prop_from_an": frac,
                    "model_prop_mean": model_props.get(anc),
                    "delta_model_minus_an": (
                        (model_props.get(anc) - frac)
                        if model_props.get(anc) is not None
                        else None
                    ),
                }
            )

anc_diag = pd.DataFrame(anc_diag_rows)
if not anc_diag.empty:
    anc_path = OUT / "lai_exp_an_global_props.tsv"
    anc_diag.to_csv(anc_path, sep="\t", index=False)
    print("wrote", anc_path)
    display(anc_diag)
else:
    print("no AN global-prop rows")

## Diagnostics: complementary haplotype switches

Among sites where **both** haps switch, fraction with A→B on hap1 and B→A on hap2
(complementary flip). Needs full `switches.tsv` (not `switches_slim`). Report for
AFR, AMR, OTH when covariates labeling is available.

In [ ]:
COMP_POPS = {"AFR", "AMR", "OTH"}
comp_rows = []


def complementary_flip_frac(sw: pd.DataFrame):
    need = {"sample", "hap", "chrom", "pos", "anc_from", "anc_to"}
    if not need.issubset(sw.columns):
        return 0, 0, None
    both = comp = 0
    for _, g in sw.groupby(["sample", "chrom", "pos"]):
        if len(g) < 2:
            continue
        h1 = g[g["hap"] == 1]
        h2 = g[g["hap"] == 2]
        if h1.empty or h2.empty:
            continue
        both += 1
        e1, e2 = h1.iloc[0], h2.iloc[0]
        if e1["anc_from"] == e2["anc_to"] and e1["anc_to"] == e2["anc_from"]:
            comp += 1
    return comp, both, (comp / both if both else None)


if pop_map is None or cmp.empty:
    print("skip complementary-flip: need covariates + scored experiments")
else:
    for _, row in exp.iterrows():
        eid = str(row[id_col])
        if ONLY_IDS and eid not in ONLY_IDS:
            continue
        if not row_is_complete(row):
            continue
        sw_full = OUT / eid / "switch_qc" / "switches.tsv.gz"
        if not sw_full.is_file():
            print("skip complementary-flip (no switches.tsv):", eid)
            continue
        sw = pd.read_csv(sw_full, sep="\t", dtype={"sample": str})
        sw = sw.merge(pop_map, left_on="sample", right_on="research_id", how="left")
        for pop in COMP_POPS:
            sub = sw[sw["population"] == pop]
            if sub.empty:
                continue
            comp, both, frac = complementary_flip_frac(sub)
            comp_rows.append(
                {
                    "experiment": eid,
                    "population": pop,
                    "n_both_hap_switch_sites": both,
                    "n_complementary": comp,
                    "complementary_frac": frac,
                }
            )

comp_df = pd.DataFrame(comp_rows)
if not comp_df.empty:
    comp_path = OUT / "lai_exp_complementary_flip.tsv"
    comp_df.to_csv(comp_path, sep="\t", index=False)
    print("wrote", comp_path)
    display(comp_df)
else:
    print("no complementary-flip rows")

## Diagnostics: FLARE model files and run logs

Parse output `.model` files with `flare_model.parse_model` when present locally
(or listed on the Terra table). Optionally tail `per_pop_log` when the table
carries log URIs.

In [ ]:
import ast

from flare_model import parse_model

model_diag_rows = []
for _, row in exp.iterrows():
    eid = str(row[id_col])
    if ONLY_IDS and eid not in ONLY_IDS:
        continue
    if not row_is_complete(row):
        continue
    run_dir = OUT / eid

    log_uris = []
    raw_log = str(row.get("per_pop_log") or "").strip()
    if raw_log:
        try:
            log_uris = ast.literal_eval(raw_log)
        except (ValueError, SyntaxError):
            log_uris = [raw_log]
    for i, uri in enumerate(log_uris):
        if uri and str(uri).startswith("gs://"):
            dest = run_dir / f"per_pop_{i}.log"
            if not dest.is_file() or FORCE_RESCAN:
                subprocess.check_call(["gsutil", "cp", str(uri), str(dest)])
            if dest.is_file():
                tail = dest.read_text().splitlines()[-5:]
                print(f"--- {eid} log tail ({dest.name}) ---")
                print("\n".join(tail))

    for model_path in sorted(run_dir.glob("*.model")):
        m = parse_model(model_path)
        model_diag_rows.append(
            {
                "experiment": eid,
                "model_file": model_path.name,
                "t_gen": m.t_gen,
                "ancestries": ",".join(m.ancestries),
                "props": ",".join(f"{p:.4f}" for p in m.props),
                "mean_miscopy": ",".join(f"{x:.4f}" for x in m.mean_miscopy()),
            }
        )

model_diag = pd.DataFrame(model_diag_rows)
if not model_diag.empty:
    md_path = OUT / "lai_exp_model_parse.tsv"
    model_diag.to_csv(md_path, sep="\t", index=False)
    print("wrote", md_path)
    display(model_diag)
else:
    print("no local .model files parsed (glob under each experiment dir)")

## Interpretation checklist

| Observation | Suggests |
|---|---|
| Pin rows: rate ≈ `expected_rate` (prop-adjusted), **flicker / hap / Mb** low | Good enough for Tractor/FELIX |
| Pin rows: rate ≫ expected, low flicker rate | Residual sustained switches (noise/heterogeneity); still may be usable |
| Pin rows: high **flicker / hap / Mb** (or high flicker fraction) | Short junk tracts remain; try stricter sites or HQ-thinned `gt` |
| `span_ok=false` | Window too short for pinned T; rates are unreliable — widen region |
| `rate_over_expected` ≫ 1 | Falsifies pinned T (monotone in T; does not select T) |
| EM rows: model T climbs toward 40–100 | Same chr1 story; do not freeze those models |
| `pin_gen_by_pop` AFR≈0.08 and AMR≈0.12 (prop-adjusted) | Per-pop pinned T working as intended |

Outputs under `OUT` (`flare_lai_exp/`):

- `flare_lai_exp.table.tsv` / `.status.tsv`
- `{experiment}/switch_qc/summary.json` (+ `switches_slim.tsv.gz`; full TSVs only if `SWITCH_QC_SUMMARY_ONLY=false`)
- `lai_exp_switch_summary.tsv`, `lai_exp_models_long.tsv`, `lai_exp_by_pop.tsv`
- `lai_exp_an_global_props.tsv`, `lai_exp_complementary_flip.tsv`, `lai_exp_model_parse.tsv`
- `lai_exp_compare.png`

Speed: default `--jobs=nproc-2` (30 on a 32-CPU VM), `--an-only`, `--summary-only`. Re-sync `scripts/` to the bucket before re-running. Set `FORCE_RESCAN=true` to replace old slow summaries.

## Part 5: per-marker rates and filter provenance

Thinning sites lowers `switches_per_hap_per_mb` mechanically. Prefer
`switches_per_hap_per_marker` and `markers_per_mb` from `flare_lai_exp.enrich_compare_row`.
Group comparisons by filter provenance; warn if a table spans more than one group.


In [ ]:
from flare_lai_exp import enrich_compare_row, warn_mixed_filter_provenance

# Example: after building per-experiment summary dicts, enrich and guard:
# enriched = [enrich_compare_row(row, summary, n_markers=n_ret) for ...]
# for w in warn_mixed_filter_provenance(enriched): print(w)
print("Part 5 helpers: enrich_compare_row, warn_mixed_filter_provenance")


## Part 8: association-facing metrics (decision rule)

Switch / tract numbers above are **diagnostics only**. Recipe choice:

1. Load / build fixed AF-divergent panels (`chr22_10mb`, `chr20_full`).
2. Score finished rows: allele–ancestry `mean_ll` + pedigree Mendelian rates.
3. Apply empirical gates (`eval_gates.json`) → survivors.
4. Tractor null-λ on survivors → winner (closest λ to 1; `mean_ll` tie-break only when CIs overlap).

See `flare/flare_lai_fix_instructions.md` Part 8. Stage panels with
`flare/scripts/stage_eval_af_panels.sh`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

import pandas as pd

from flare_lai_exp import (
    apply_eval_gates,
    enrich_association_scores,
)
from flare_lai_null_lambda import pick_winner

BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
PANEL_DIR = Path(os.environ.get("FLARE_EVAL_PANEL_DIR", str(OUT / "eval_panels")))
PANEL_DIR.mkdir(parents=True, exist_ok=True)
GATES_PATH = Path(os.environ.get("FLARE_EVAL_GATES", str(PANEL_DIR / "eval_gates.json")))
ASSOC_OUT = OUT / "association_scores"
ASSOC_OUT.mkdir(parents=True, exist_ok=True)

# Prefer staged GCS panels when present
for label in ("chr22_10mb", "chr20_full"):
    local = PANEL_DIR / f"{label}.markers.tsv"
    if local.is_file():
        continue
    if BUCKET:
        remote = f"{BUCKET}/refs/flare/eval_panels/{label}.markers.tsv"
        try:
            subprocess.check_call(["gsutil", "cp", remote, str(local)])
        except Exception as exc:
            print("panel not staged yet:", label, exc)

print("PANEL_DIR:", PANEL_DIR)
print("panels:", sorted(PANEL_DIR.glob("*.markers.tsv")))
print("GATES_PATH:", GATES_PATH, "exists=", GATES_PATH.is_file())

# Map each finished row to a panel by chromosome of its region
def panel_for_region(region: str) -> Path | None:
    chrom = str(region).split(":")[0]
    if chrom == "chr22":
        cand = PANEL_DIR / "chr22_10mb.markers.tsv"
    elif chrom == "chr20":
        cand = PANEL_DIR / "chr20_full.markers.tsv"
    else:
        return None
    return cand if cand.is_file() else None

PED = Path(os.environ.get(
    "AOU_PHASE2_PED",
    str(Path("tractor_mix/resources/legacy_covariates/aou_phase2.ped")),
))
# Genetic map: override per chrom as needed
MAP_TMPL = os.environ.get(
    "FLARE_MAP_TMPL",
    "maps/plink.chrchr{chrom_num}.GRCh38.map",
)

assoc_rows = []
if "cmp" in dir() and not cmp.empty:
    for _, row in cmp.iterrows():
        eid = str(row.get(DEFAULT_ID_COLUMN) or row.get("experiment") or "")
        region = str(row.get("region") or "")
        anc = row.get("anc_vcf") or row.get("local_anc_vcf")
        gt = row.get("gt_vcf") or os.environ.get("FLARE_GT_VCF", "")
        panel = panel_for_region(region)
        if not panel or not anc or not gt:
            print("skip association score (missing panel/anc/gt):", eid)
            continue
        run_dir = ASSOC_OUT / eid
        run_dir.mkdir(parents=True, exist_ok=True)
        allele_json = run_dir / "allele_ancestry_score.json"
        mendel_json = run_dir / "mendelian_lai_score.json"
        if not allele_json.is_file() or os.environ.get("FORCE_ASSOC_RESCORE") == "true":
            cmd = [
                "python3", str(SCRIPTS / "flare_score_allele_ancestry.py"),
                "--anc-vcf", str(anc),
                "--gt-vcf", str(gt),
                "--panel", str(panel),
                "--region", region,
                "--experiment", eid,
                "--out", str(allele_json),
            ]
            samples = row.get("analysis_samples") or os.environ.get("ANALYSIS_SAMPLES", "")
            if samples:
                cmd.extend(["--samples", str(samples)])
            print("+", " ".join(cmd))
            subprocess.check_call(cmd)
        chrom = region.split(":")[0]
        chrom_num = chrom.replace("chr", "")
        map_path = Path(MAP_TMPL.format(chrom_num=chrom_num, chrom=chrom))
        if map_path.is_file() and PED.is_file() and (
            not mendel_json.is_file() or os.environ.get("FORCE_ASSOC_RESCORE") == "true"
        ):
            cmd = [
                "python3", str(SCRIPTS / "flare_score_mendelian_lai.py"),
                "--anc-vcf", str(anc),
                "--ped", str(PED),
                "--map", str(map_path),
                "--panel", str(panel),
                "--region", region,
                "--experiment", eid,
                "--out", str(mendel_json),
            ]
            cov = os.environ.get("COVARIATES_PATH", "")
            if cov:
                cmd.extend(["--covariates", cov])
            print("+", " ".join(cmd))
            subprocess.check_call(cmd)
        allele = json.loads(allele_json.read_text()) if allele_json.is_file() else {}
        mendel = json.loads(mendel_json.read_text()) if mendel_json.is_file() else {}
        assoc_rows.append(enrich_association_scores(row, allele_json=allele, mendel_json=mendel))
else:
    print("cmp empty — run switch-QC section first")

assoc_df = pd.DataFrame(assoc_rows)
if not assoc_df.empty:
    display(assoc_df)
    assoc_df.to_csv(OUT / "lai_exp_association_scores.tsv", sep="\t", index=False)
    print("wrote", OUT / "lai_exp_association_scores.tsv")

# Gates: write template if missing; apply when numeric thresholds present
if not GATES_PATH.is_file():
    template = {
        "_comment": "Set numeric thresholds after inspecting assoc_df distributions",
        "min_mean_ll": None,
        "max_violations_per_informative_locus": None,
        "max_excess_recomb_over_expected": None,
    }
    GATES_PATH.write_text(json.dumps(template, indent=2) + "\n")
    print("wrote template gates:", GATES_PATH)

gates = json.loads(GATES_PATH.read_text())
gates_active = {k: v for k, v in gates.items() if not str(k).startswith("_") and v is not None}
if gates_active and not assoc_df.empty:
    survivors, eliminated = apply_eval_gates(assoc_df.to_dict(orient="records"), gates_active)
    print(f"survivors={len(survivors)} eliminated={len(eliminated)} gates={gates_active}")
    display(pd.DataFrame(survivors))
    (ASSOC_OUT / "survivors.json").write_text(json.dumps(survivors, indent=2) + "\n")
else:
    survivors = assoc_df.to_dict(orient="records") if not assoc_df.empty else []
    print("no active gates yet — inspect mean_ll / Mendelian distributions, then edit", GATES_PATH)

# Part 4 null-λ: load per-survivor scores if present; declare winner
lambda_rows = []
for r in survivors:
    eid = r.get("experiment") or ""
    lam_path = ASSOC_OUT / eid / "null_lambda_score.json"
    if lam_path.is_file():
        lam = json.loads(lam_path.read_text())
        enriched = enrich_association_scores(r, lambda_json=lam)
        r = {**r, **enriched}
        lambda_rows.append(r)
    else:
        print(
            "null-λ not run for", eid,
            "— use scripts/flare_lai_null_lambda.py on survivors (Tractor docker + GRM)",
        )

if lambda_rows:
    decision = pick_winner(lambda_rows)
    print(json.dumps(decision, indent=2))
    (ASSOC_OUT / "winner.json").write_text(json.dumps(decision, indent=2) + "\n")
    display(pd.DataFrame(lambda_rows))
else:
    print("Part 4: run null-λ on survivors when Tractor inputs are ready; winner deferred.")
